# OpenAI Agents SDK - Brief Demo (FOR DEMO)

Quick demonstration of five key concepts in the OpenAI Agents SDK:
1. **Agents** - Defining specialized AI agents
2. **Runner.run()** - Executing an agent and inspecting its results
3. **Handoffs** - Agents transferring control to others
4. **Guardrails** - Prompt-based restrictions and SDK input validation
5. **Sessions** - Stateful conversation history


In [1]:
# Set up
%pip install -q python-dotenv openai-agents

In [2]:
import os
import json
from openai import OpenAI

def load_openai_key():
    """Read the OpenAI API key from Colab Secrets; ask for it only if that fails."""
    try:
        from google.colab import userdata
        return userdata.get("OPENAI_API_KEY")
    except ImportError:
        print("Not running in Colab.")
    except Exception as e:
        # e.g. the secret is missing, or Notebook access is turned off
        print(f"Could not read the Colab secret OPENAI_API_KEY ({type(e).__name__}).")
    from getpass import getpass
    return getpass("Enter your OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = load_openai_key()
#client = OpenAI()

## 1. Agents - Define and Run Specialized AI Agents

In [3]:
from agents import Agent, Runner, SQLiteSession
import asyncio

# Create specialized agents with specific instructions
MODEL = "gpt-4.1-mini"

# Define math_agent
math_agent = Agent(
    name="Math Agent",
    instructions="You solve math problems step by step. Be concise.",
    model=MODEL,
)
# Define history_agent
history_agent = Agent(
    name="History Agent",
    instructions="You answer history questions with dates and context. Be concise.",
    model=MODEL,
)

# Actually run the agents, run #1 -> math agent, run #2 -> history agent
# It is Asynchronous Sequential Execution
async def demo_agents():
    # Math agent
    math_result = await Runner.run(math_agent, "What is 5 + 3?")
    print(f"Math Agent: {math_result.final_output}\n")

    # History agent
    history_result = await Runner.run(history_agent, "When was World War II?")
    print(f"History Agent: {history_result.final_output}")

await demo_agents()

Math Agent: 5 + 3 = 8

History Agent: World War II lasted from 1939 to 1945. It began on September 1, 1939, with Germany's invasion of Poland, and ended on September 2, 1945, with Japan's formal surrender.


## 2. Runner.run() - Execute an Agent

An `Agent` defines the instructions and capabilities; **`Runner.run()` executes it**. In Google Colab, use `await Runner.run(...)` because the notebook supports asynchronous Python.

**Execution flow (simple case):**

1. `Runner.run(agent, input)` starts an agent run.
2. The SDK sends the agent's instructions and user input to the configured model.
3. The model generates a response. If a handoff or tool call is requested, the runner handles it and continues the loop (subject to limits).
4. The run ends when the active agent produces a final output, or stops due to an error/limit/guardrail tripwire.
5. The returned `RunResult` contains the final answer and execution details.

**Useful result attributes:**

| Attribute | Meaning |
| --- | --- |
| `result.final_output` | Final answer from the run |
| `result.last_agent` | Agent that was active at the end |
| `result.new_items` | Items generated during this run, such as messages or handoffs |
| `result.raw_responses` | Model response objects collected during the run |
| `result.input` | Input provided to the run |

**Important distinctions:** One `Runner.run()` can make **multiple LLM calls** (for example, a triage agent hands off to a specialist). A separate nested `Runner.run()` can also occur inside a guardrail. `Runner.run()` is asynchronous; `Runner.run_sync()` is the synchronous alternative for regular Python scripts, but should not be used inside an already-running event loop.

The next code cell inspects a `RunResult` returned by `Runner.run()`.


In [4]:
# Inspect the model response object
result = await Runner.run(
    history_agent,
    "Who was the first president of the United States?"
)
# input
print("INPUT:")
print(result.input)

# final output
print("\nFINAL OUTPUT:")
print(result.final_output)

# new events generated by this run
print("\nNEW ITEMS:")
print(result.new_items)

# raw model response
print("\nRAW RESPONSES:")
print(result.raw_responses)

# convert the raw response into JSON. Note: the raw_response is the model-level response wrapper by Agents SDK.
from pprint import pprint
print("\nRAW RESPONSES AS JSON:")
pprint(result.raw_responses[0])

# last agent was activated
print("\nLAST AGENT:")
print(result.last_agent.name)


INPUT:
Who was the first president of the United States?

FINAL OUTPUT:
The first president of the United States was George Washington. He served from April 30, 1789, to March 4, 1797.

NEW ITEMS:
[MessageOutputItem(agent=Agent(name='History Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You answer history questions with dates and context. Be concise.', prompt=None, handoffs=[], model='gpt-4.1-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None, prompt_cache_options=None, preserve_raw_usage=None, timeout=None), input_guardrails=[], output_guardrails=[], o

## 3. Handoffs - Agents Transfer Control

3.1. Define Triage Agent

In [5]:
# Triage agent can hand off to specialized agents
triage_agent = Agent(
    name="Triage Agent",
    instructions="Route math questions to Math Agent, history to History Agent. Briefly say who you're routing to.",
    handoffs=[math_agent, history_agent],
    model=MODEL,
)

3.2 Demonstrate Handoff to Math Agent

In [6]:
# Demonstrate actual handoff, run #1 -> triage -> math agent; run #2->triage -> history agent
async def demo_handoff():
    result = await Runner.run(triage_agent, "What is 10 divided by 2?")
    print(result.last_agent)
    print(f"Triage: {result.final_output}")
await demo_handoff()

Agent(name='Math Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You solve math problems step by step. Be concise.', prompt=None, handoffs=[], model='gpt-4.1-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None, prompt_cache_options=None, preserve_raw_usage=None, timeout=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)
Triage: 10 divided by 2 is calculated as:

\( \frac{10}{2} = 5 \).

So, the answer is 5.


3.2 Demonstrate Handoff to History Agent

In [7]:
# Demonstrate actual handoff
async def demo_handoff2():
    result = await Runner.run(triage_agent, "When did Singapore gain independence?")
    print(result.last_agent)
    print(f"Triage: {result.final_output}")
await demo_handoff2()

Agent(name='History Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You answer history questions with dates and context. Be concise.', prompt=None, handoffs=[], model='gpt-4.1-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None, prompt_cache_options=None, preserve_raw_usage=None, timeout=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)
Triage: Singapore gained independence on August 9, 1965, when it separated from Malaysia and became a sovereign nation. This followed 

## 4. Guardrails - Input Validation (Optional)

### 4.1 prompt-based restriction

In [8]:
# Create an agent with validation built into instructions
# This demonstrates guardrail concept through agent behavior
restricted_agent = Agent(
    name="Restricted Agent",
    instructions="""You ONLY answer math or history questions.
    If asked about anything else (weather, sports, food, etc.), respond:
    'I can only help with math or history questions.'"""
)
# run #1-> math, run #2->history, run #3 -> weather
async def demo_guardrails():
    print("Testing input validation through agent behavior:\n")

    # Valid: Math question
    result = await Runner.run(restricted_agent, "What is 5 + 3?")
    print(f"✅ Math question: {result.final_output}\n")

    # Valid: History question
    result = await Runner.run(restricted_agent, "When was the American Revolution?")
    print(f"✅ History question: {result.final_output}\n")

    # Invalid: Weather question (agent should refuse)
    result = await Runner.run(restricted_agent, "What's the weather today?")
    print(f"❌ Weather question: {result.final_output}")

await demo_guardrails()

Testing input validation through agent behavior:

✅ Math question: 8

✅ History question: The American Revolution lasted from 1775 to 1783.

❌ Weather question: I can only help with math or history questions.


### 4.2 Real SDK Input
This version uses a real SDK **input guardrail**:

```text
User Input
    |
    v
Input Guardrail
    |
    +-- allowed --> Restricted Agent --> Answer
    |
    +-- blocked --> Tripwire --> Execution stops
```

The guardrail uses a small classification agent to determine whether the user's request is about **math or history**. If it is not, `tripwire_triggered=True` causes `InputGuardrailTripwireTriggered`.

`run_in_parallel=False` is used so the guardrail finishes **before** the protected agent starts.

In [9]:
from pydantic import BaseModel

from agents import (
    Agent,
    Runner,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    RunContextWrapper,
    TResponseInputItem,
    input_guardrail,
)

class TopicCheck(BaseModel):
    is_allowed: bool
    reason: str

# A dedicated agent evaluates whether the input is in scope.
guardrail_agent = Agent(
    name="Math or History Guardrail",
    instructions=(
        "Determine whether the user's request is about mathematics or history. "
        "Set is_allowed=true only for math or history questions. "
        "Set is_allowed=false for other topics such as weather, sports, food, travel, etc."
    ),
    output_type=TopicCheck,
    model=MODEL,
)

@input_guardrail(run_in_parallel=False)
async def math_history_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:

    check = await Runner.run(
        guardrail_agent,
        input,
        context=ctx.context,
    )

    return GuardrailFunctionOutput(
        output_info=check.final_output,
        tripwire_triggered=not check.final_output.is_allowed,
    )

restricted_agent = Agent(
    name="Restricted Agent",
    instructions=(
        "Answer math and history questions clearly and concisely. "
        "Input scope is enforced by an SDK input guardrail."
    ),
    model=MODEL,
    input_guardrails=[math_history_guardrail],  # ✅ CORRECTED
)
async def run_guardrail_test(label: str, question: str):
    print(f"{label}: {question}")

    try:
        result = await Runner.run(restricted_agent, question)
        print(f"✅ Allowed: {result.final_output}\n")

    except InputGuardrailTripwireTriggered as exc:
        info = exc.guardrail_result.output.output_info
        print("🛑 Blocked by input guardrail")
        print(f"Reason: {info.reason}\n")
async def demo_guardrails():
    await run_guardrail_test(
        "Math question",
        "What is 5 + 3?",
    )

    await run_guardrail_test(
        "History question",
        "When was the American Revolution?",
    )

    await run_guardrail_test(
        "Weather question",
        "What's the weather today?",
    )


await demo_guardrails()


Math question: What is 5 + 3?
✅ Allowed: 5 + 3 = 8

History question: When was the American Revolution?
✅ Allowed: The American Revolution took place from 1775 to 1783.

Weather question: What's the weather today?
🛑 Blocked by input guardrail
Reason: The question is about weather, which is not related to mathematics or history.



Discussion (Optional)

How many runners for each example?

Solution: run_guardrail_test() starts Run #1 through Runner.run(restricted_agent, ...). During Run #1, the input guardrail is executed, and that input guardrail starts Run #2 through Runner.run(guardrail_agent, ...)"-> check = await Runner.run(
        guardrail_agent,..."

## 5. Sessions - Stateful Conversation History

In [11]:
# Sessions maintain conversation history across interactions
# Stateful conversation within the current process/session
async def demo_sessions():
    # Create a session for maintaining state
    session = SQLiteSession("demo_conversation") # it is in memory SQLite

    print("With Sessions - Stateful conversation:\n")

    # First interaction, run #1
    result1 = await Runner.run(
        math_agent,
        "Calculate 4 * 4",
        session=session
    )
    print(f"Q1: Calculate 4 * 4")
    print(f"A1: {result1.final_output}\n")

    # Second interaction - agent remembers previous context, run #2
    result2 = await Runner.run(
        math_agent,
        "Now multiply that result by 3",
        session=session
    )
    print(f"Q2: Now multiply that result by 3")
    print(f"A2: {result2.final_output}")
    print("\n✅ Session maintains conversation history!")

await demo_sessions()

With Sessions - Stateful conversation:

Q1: Calculate 4 * 4
A1: 4 * 4 = 16

Q2: Now multiply that result by 3
A2: 16 * 3 = 48

✅ Session maintains conversation history!


## Summary

**OpenAI Agents SDK** demonstrated:
- **Agents**: Define specialized AI agents with instructions and a model
- **Runner.run()**: Execute an agent, manage the agent loop, and inspect `RunResult`
- **Handoffs**: Transfer control from a triage agent to a specialist
- **Guardrails**: Contrast prompt-based restrictions with an SDK input guardrail
- **Sessions**: Preserve conversation history using `SQLiteSession`

The demo focuses on these concepts and does not include function tools.
